### Overview
This notebook demonstrates an end-to-end, secure data processing pipeline:
1. **Ingest:** Extract encrypted trace logs from Google BigQuery.
2. **Decrypt:** Perform in-memory symmetric decryption locally.
3. **Analyze:** Classify sentiment using a local, air-gapped LLM via Ollama—ensuring sensitive trace data never leaves the local machine.
4. **Persist:** Write enriched results back to BigQuery for downstream reporting.

---
## 1. Environment & Dependency Setup
Installing required libraries for BigQuery I/O, symmetric encryption, and local LLM orchestration.

In [38]:
# Install required packages
%pip install ollama pandas google-cloud-bigquery cryptography db-dtypes


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:

# BIGQUERY
PROJECT_ID = "analytics-cologne"
CREDENTIAL_PARTH = "service_account.json"
DATASET = "ap_meetup_cologne"
DEMO_TABLE = "ap_meetup_demo"
RESULT_TABLE = "sentiment_results"

# ENCRYPTION
ENCRYPTION_KEY = b'5og3EDC0SFn0A7Gnz6HKXxZXEOcywv7QrV2BzTy74gE='

# LLM
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "llama3"

## 2. Data Ingestion from BigQuery
Pulling a batch of raw, encrypted records from our BigQuery warehouse into a local Pandas DataFrame.

In [40]:
import pandas_gbq
import pandas as pd
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(CREDENTIAL_PARTH)

query = f"""
    SELECT 
        chat_id,
        event_id,
        TIMESTAMP(timestamp) AS timestamp,
        event_name,
        encrypted_chat_body
    FROM 
        `{PROJECT_ID}.{DATASET}.{DEMO_TABLE}`
    WHERE
        event_name IN ("receive_response", "submit_message")
    ORDER BY 
        chat_id ASC,
        TIMESTAMP(timestamp) ASC
"""

df = pandas_gbq.read_gbq(query, project_id = PROJECT_ID, credentials = credentials)
print(f"Loaded {len(df)} rows successfully.")

df.head()

Loaded 12 rows successfully.


/Users/ramonozod-seradj/Desktop/VS_Code/analytics_meetup_20261125/venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2926: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,chat_id,event_id,timestamp,event_name,encrypted_chat_body
0,chat_90421,evt_1002,2026-11-20 09:14:15.482+00:00,submit_message,gAAAAABrACWGw9vyPhmKpJy5jAMo0-jMt6C14xF0jjVkZs...
1,chat_90421,evt_1003,2026-11-20 09:14:18.905+00:00,receive_response,gAAAAABrACWGR2H8SjN5A4SSjcGzxLI1aiylSeHnZZs2DI...
2,chat_90421,evt_1004,2026-11-20 09:14:42.310+00:00,submit_message,gAAAAABrACWGf-KbtGUnLpxq2p5F47ZwcCAeVsijzz1TJy...
3,chat_90421,evt_1005,2026-11-20 09:14:46.120+00:00,receive_response,gAAAAABrACWGfCkp3YWOA3Vya7ABMV2NWK0WBrghkpNpBd...
4,chat_90421,evt_1006,2026-11-20 09:15:03.774+00:00,submit_message,gAAAAABrACWGt7JPOmYZLrb9IpLjk3zVzvbpzkH3AsiDqc...


## 3. In-Memory Symmetric Decryption
Using **Fernet (AES-128-CBC)** symmetric encryption to unlock the `trace` payload locally. Decrypted text exists strictly in local runtime memory and is never sent to external third-party APIs.

In [41]:
from cryptography.fernet import Fernet

cipher_suite = Fernet(ENCRYPTION_KEY)

def decrypt_text(encrypted_value):
    if pd.isna(encrypted_value):
        return None
    try:
        if isinstance(encrypted_value, str):
            encrypted_value = encrypted_value.encode("utf-8")
            decrypted_bytes = cipher_suite.decrypt(encrypted_value)
        return decrypted_bytes.decode("utf-8")
    except Exception as e:
        return f"[Decryption Error: {e}]"

df["chat_body"] = df["encrypted_chat_body"].apply(decrypt_text)

df["user_chat"] = df["chat_body"].where(
    df["event_name"] == "submit_message"
)

df["llm_chat"] = df["chat_body"].where(
    df["event_name"] == "receive_response"
)

result_df = (
    df.groupby("chat_id", sort=False)
    .agg(
        chat_body = ("chat_body", lambda s: [x for x in s if pd.notna(x)]),
        user_chat = ("user_chat", lambda s: [x for x in s if pd.notna(x)]),
        llm_chat = ("llm_chat", lambda s: [x for x in s if pd.notna(x)])
    )
    .reset_index()
)

result_df.head(10)

,chat_id,chat_body,user_chat,llm_chat
0,chat_90421,"[Hallo, ich hatte gerade auf der Autobahn eine...","[Hallo, ich hatte gerade auf der Autobahn eine...",[Das tut mir leid zu hören! Ich helfe Ihnen ge...
1,chat_90422,"[Hi, ich fliege nächste Woche in die USA. Bin ...","[Hi, ich fliege nächste Woche in die USA. Bin ...",[Guten Tag! Ich habe in Ihren Vertragsdaten na...


## 4. Local Inference Engine Verification
Before processing data, we verify that the local **Ollama** daemon is active and serving models on `localhost:11434`. If the service is dormant, the script automatically initializes it in the background.

In [42]:
import subprocess
import requests
from IPython.display import HTML, display

try:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
    if response.status_code == 200:
        models = [m["name"] for m in response.json().get("models", [])]
        display(
            HTML(
                "<b style='color: green;'>✔ Ollama is running!</b> Available"
                f" models: {models}"
            )
        )
    else:
        raise Exception()
except Exception:
    display(
        HTML(
            "<b style='color: orange;'>⚠️ Ollama is not running. Launching"
            " app...</b>"
        )
    )
    try:
        subprocess.run(["open", "-a", "Ollama"], check=True)
        import time

        time.sleep(4)
        display(
            HTML(
                "<b style='color: green;'>✔ Ollama launched successfully.</b>"
            )
        )
    except Exception as e:
        display(
            HTML(
                f"<b style='color: red;'>❌ Could not auto-launch Ollama: {e}</b>"
            )
        )

## 5. Local LLM Sentiment Classification
Passing the decrypted traces to our local `llama3.2` model with deterministic parameters (`temperature=0.0`) to classify each entry as **Positive**, **Negative**, or **Neutral**.

In [46]:
import ollama

def analyze_sentiment(text):
    if not text or "[Decryption Error" in text:
        return "Skipped"

    prompt = (
        "Analysiere des Sentiment des folgenden LLM Chatbot Traces."
        "Antworte mit nur einem Wort: Positiv, Negativ, oder Neutral.\n\n"
        f"Trace: {text}"
    )

    try:
        response = ollama.chat(
            model = MODEL_NAME,
            messages=  [{"role": "user", "content": prompt}],
            options = {"temperature": 0.0},  # Low temp for more deterministic output
        )
        return response["message"]["content"].strip()
    except Exception as e:
        return f"Error: {e}"

def classify_contents(text):
    if not text or "[Decryption Error" in text:
        return "Skipped"

    prompt = (
        "Klassifiziere den Inhalt des folgenden LLM Chatbot Traces. "
        "Antworte mit nur einem Wort: Sales, Kundenservice oder Sonstiges.\n\n"
        f"Trace: {text}"
    )

    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.0},  # Low temp for more deterministic output
        )
        return response["message"]["content"].strip()
    except Exception as e:
        return f"Error: {e}"

# Run sentiment analysis on the decrypted traces
print("Analyzing sentiments with Ollama...")
result_df["sentiment"] = result_df["chat_body"].apply(analyze_sentiment)
result_df["category"] = result_df["chat_body"].apply(classify_contents)

# View summary of results
print("\nSentiment Distribution:")
print(result_df["sentiment"].value_counts())

# Show a sample of the final results
result_df[["chat_body", "sentiment", "category"]].head(10)

Analyzing sentiments with Ollama...

Sentiment Distribution:
sentiment
Neutral    1
Positiv    1
Name: count, dtype: int64


,chat_body,sentiment,category
0,"[Hallo, ich hatte gerade auf der Autobahn eine...",Neutral,Kundenservice
1,"[Hi, ich fliege nächste Woche in die USA. Bin ...",Positiv,Kundenservice


## 6. Warehouse Persistence
Appending the enriched dataset—now containing sentiment classifications—back to our destination table in BigQuery.

In [ ]:
from datetime import datetime
import pandas_gbq

result_df["processing_date"] = datetime.now()
upload_df = result_df.drop(columns = ["chat_body", "user_chat", "llm_chat"])
table_id = f"{DATASET}.{RESULT_TABLE}"

pandas_gbq.to_gbq(
    upload_df,
    project_id = PROJECT_ID, 
    credentials = credentials, 
    destination_table = table_id, 
    if_exists = "append"
)

print(f"Success! Loaded result rows into {table_id}.")
upload_df.head()

Success! Loaded result rows into ap_meetup_cologne.sentiment_results.


,chat_id,sentiment,category,processing_date
0,chat_90421,Neutral,Vertrieb,2026-11-20 12:52:33.751104
1,chat_90422,Positiv,Vertrieb,2026-11-20 12:52:33.751104
